# Generation and pairwise files
Run in two sessions:
- **Part 1** (after SFT): sanity check, then base and SFT generations.
- **Part 2** (after `dpo_hh`, `dpo_uf` and `lora_persona`): DPO and LoRA generations, then the pairwise files.

Each generation is skipped if its output file already exists. For Part 2, attach this notebook's saved Part 1 output and set `PREVIOUS_EVAL` so finished files are reused.

## Setup
Set `GITHUB_USER` below. The repository is private, so add a Kaggle secret named `GITHUB_TOKEN` (a GitHub personal access token with read access) under *Add-ons → Secrets*. The token is passed as an HTTP header and is not written into the cloned repository.

In [ ]:
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

os.environ["PYTHONUNBUFFERED"] = "1"

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Small-Model_Alignment"
REPO_DIR = f"/kaggle/working/{REPO_NAME}"

if not os.path.isdir(REPO_DIR):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    credentials = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()
    clone_command = [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credentials}",
        "clone",
        "-q",
        f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
        REPO_DIR,
    ]
    result = subprocess.run(clone_command)
    if result.returncode != 0:
        raise SystemExit("git clone failed")

os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "trl==1.16.0"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi

In [ ]:
import datasets
import peft
import torch
import transformers
import trl

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if torch.cuda.device_count() < 2:
    raise SystemExit("Expected two GPUs. Select the GPU T4 x2 accelerator.")

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

## Inputs
Set the folders that are attached for this session. Leave a path as it is if that input is not attached yet.

In [ ]:
import shutil

SFT_OUTPUTS = "/kaggle/input/YOUR-SFT-NOTEBOOK-OUTPUT/Small-Model_Alignment/outputs"
DPO_HH_OUTPUTS = "/kaggle/input/YOUR-DPO-HH-NOTEBOOK-OUTPUT/Small-Model_Alignment/outputs"
DPO_UF_OUTPUTS = "/kaggle/input/YOUR-DPO-UF-NOTEBOOK-OUTPUT/Small-Model_Alignment/outputs"
LORA_OUTPUTS = "/kaggle/input/YOUR-LORA-NOTEBOOK-OUTPUT/Small-Model_Alignment/outputs"
PREVIOUS_EVAL = ""

os.makedirs("outputs/eval", exist_ok=True)
if PREVIOUS_EVAL != "":
    if not os.path.isdir(PREVIOUS_EVAL):
        raise SystemExit(f"PREVIOUS_EVAL does not exist: {PREVIOUS_EVAL}")
    shutil.copytree(PREVIOUS_EVAL, "outputs/eval", dirs_exist_ok=True)
print(sorted(os.listdir("outputs/eval")))

## Part 1 — sanity check (3 prompts)
Read these outputs before the full runs: checkpoint path, prompt format, EOS behaviour and length. If outputs loop, set `repetition_penalty: 1.1` in `configs/generation.yaml` and use it for every model in that comparison.

In [ ]:
!python scripts/generate.py --config configs/generation.yaml --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-base --output_path outputs/eval/sanity/base_alpaca_eval.json --sanity 3
!python scripts/generate.py --config configs/generation.yaml --model_path {SFT_OUTPUTS}/sft/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output_path outputs/eval/sanity/sft_alpaca_eval.json --sanity 3
!python scripts/generate.py --config configs/generation.yaml --model_path {SFT_OUTPUTS}/sft/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output_path outputs/eval/sanity/sft_hh.json --sanity 3

## Part 1 — base and SFT generations

In [ ]:
!python scripts/generate.py --config configs/generation.yaml --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-base --output_path outputs/eval/base_alpaca_eval.json
!python scripts/generate.py --config configs/generation.yaml --model_path {SFT_OUTPUTS}/sft/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output_path outputs/eval/sft_alpaca_eval.json
!python scripts/generate.py --config configs/generation.yaml --model_path {SFT_OUTPUTS}/sft/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output_path outputs/eval/sft_hh.json

## Part 2 — sanity check for the DPO and LoRA models (3 prompts)

In [ ]:
!python scripts/generate.py --config configs/generation.yaml --model_path {DPO_HH_OUTPUTS}/dpo_hh/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh --output_path outputs/eval/sanity/dpo_hh_hh.json --sanity 3
!python scripts/generate.py --config configs/generation.yaml --model_path {DPO_UF_OUTPUTS}/dpo_uf/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-uf --output_path outputs/eval/sanity/dpo_uf_alpaca_eval.json --sanity 3
!python scripts/generate.py --config configs/generation.yaml --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path {LORA_OUTPUTS}/lora_persona/best --eval_set eval_sets/persona_style_eval_prompts.json --generator smollm2-360m-lora-persona --output_path outputs/eval/sanity/lora_persona.json --sanity 3

## Part 2 — DPO and LoRA generations

In [ ]:
!python scripts/generate.py --config configs/generation.yaml --model_path {DPO_HH_OUTPUTS}/dpo_hh/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh --output_path outputs/eval/dpo_hh_hh.json
!python scripts/generate.py --config configs/generation.yaml --model_path {DPO_UF_OUTPUTS}/dpo_uf/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-uf --output_path outputs/eval/dpo_uf_alpaca_eval.json
!python scripts/generate.py --config configs/generation.yaml --model_path HuggingFaceTB/SmolLM2-360M-Instruct --eval_set eval_sets/persona_style_eval_prompts.json --generator smollm2-360m-instruct --output_path outputs/eval/smollm2_base_persona.json
!python scripts/generate.py --config configs/generation.yaml --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path {LORA_OUTPUTS}/lora_persona/best --eval_set eval_sets/persona_style_eval_prompts.json --generator smollm2-360m-lora-persona --output_path outputs/eval/lora_persona.json

## Part 2 — pairwise files
Candidate vs baseline, seeded A/B order (seed 42). Written once, before any judging.

In [ ]:
!python scripts/build_pairs.py --candidate_path outputs/eval/dpo_hh_hh.json --baseline_path outputs/eval/sft_hh.json --output_path outputs/eval/dpo_hh_vs_sft_hh_pairs.jsonl
!python scripts/build_pairs.py --candidate_path outputs/eval/dpo_uf_alpaca_eval.json --baseline_path outputs/eval/sft_alpaca_eval.json --output_path outputs/eval/dpo_uf_vs_sft_alpaca_eval_pairs.jsonl
!python scripts/build_pairs.py --candidate_path outputs/eval/lora_persona.json --baseline_path outputs/eval/smollm2_base_persona.json --output_path outputs/eval/lora_vs_base_persona_pairs.jsonl

## Archive

In [ ]:
!cd outputs && zip -q -r /kaggle/working/generation_outputs.zip eval
!ls -lh /kaggle/working/generation_outputs.zip